In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import sys
sys.path.append('C:/Users/inesm/projectos/tennis-predictor/src/data/')
from make_dataset import load_data
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import yaml 
import pickle

with open("C:/Users/inesm/projectos/tennis-predictor/conf/conf.yaml", "r") as config_file:
    conf = yaml.safe_load(config_file)
    
# Load data function
def load_train_data():

    input_filepath = "C:/Users/inesm/projectos/tennis-predictor/data/processed/"

    # Load data using the load_data function
    dataset = load_data(input_filepath, file_name='features_classified.pkl')

    # load auxiliar data to use in the custom profit function
    aux_dataset = load_data(input_filepath, file_name='features.pkl')

    # Define training datasets
    y = dataset["winner_is_p1"]    
    features_names = conf['classified_features']
    X = dataset[features_names]

    return y, X

# Make predictions function
def make_prediction(odd1, odd2, rank1, rank2, P1_wins, P2_wins):
    
    # Load train data
    y, X = load_train_data()

    # classifier best parameters
    best_params = conf['random-forest']['best-params']

    # Additional parameters
    additional_params = {
        'class_weight': conf['training']['class_weights'],
        'random_state': 42
    }

    # Combine best_params and additional_params to create the final parameters
    final_params = {**best_params, **additional_params}

    # Create a decision tree classifier with the best parameters
    classifier = RandomForestClassifier(**final_params)

    # Train the model on the training data
    classifier.fit(X, y)

    input = {
    'OddP1': [odd1],
    'RankP2': [rank2],
    'H2H': [int(P1_wins)-int(P2_wins)]
    }

    input = pd.DataFrame(input)
    # Make predictions on the test data
    y_pred = classifier.predict(input)

    if y_pred == 1:
        return 'Bet on P1'
    else:
        return 'Bet on P2'

# Define widgets
odd1_widget = widgets.Text(description="1st Player Odd:")
odd2_widget = widgets.Text(description="2nd Player Odd:")
rank1_widget = widgets.Text(description="1st Player Rank:")
rank2_widget = widgets.Text(description="2nd Player Rank:")
P1_wins_widget = widgets.Text(description="P1 Wins over P2:")
P2_wins_widget = widgets.Text(description="P2 Wins over P1:")
predict_button = widgets.Button(description="Make Prediction")
output_widget = widgets.Output()

# Define function to be called on button click
def on_predict_button_click(b):
    with output_widget:
        clear_output()
        odd1 = odd1_widget.value
        odd2 = odd2_widget.value
        rank1 = rank1_widget.value
        rank2 = rank2_widget.value
        P1_wins = P1_wins_widget.value
        P2_wins = P2_wins_widget.value

        # Validation (add your own validation logic)
        if not all(value.replace('.', '', 1).isdigit() for value in [odd1, odd2]):
            print("Please enter valid numbers for odds.")
            return
        if not all(value.isdigit() for value in [rank1, rank2, P1_wins, P2_wins]):
            print("Please enter valid numbers for ranks and wins.")
            return
                
        # Make prediction
        result = make_prediction(odd1, odd2, rank1, rank2, P1_wins, P2_wins)
        print(result)

# Attach the function to the button's click event
predict_button.on_click(on_predict_button_click)

# Display widgets
display(
    odd1_widget, odd2_widget,
    rank1_widget, rank2_widget,
    P1_wins_widget, P2_wins_widget,
    predict_button, output_widget
)

Text(value='', description='1st Player Odd:')

Text(value='', description='2nd Player Odd:')

Text(value='', description='1st Player Rank:')

Text(value='', description='2nd Player Rank:')

Text(value='', description='P1 Wins over P2:')

Text(value='', description='P2 Wins over P1:')

Button(description='Make Prediction', style=ButtonStyle())

Output()